In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd

from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA

sys.path.append("../../utils")

from utils import (
    cargar_dataset,
    guardar_dataset_csv,
    resumen_clases
)

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# =========================
# CONFIGURACIÓN GENERAL
# =========================

DATASET_NAME = "CIC17"

LABEL_COL = "LABEL"

# Dataset de entrada
INPUT_DATASET_VERSION = "CIC17__split__v1"
INPUT_DIR = f"../../../02_datasets/processed/{INPUT_DATASET_VERSION}"

TRAIN_FILENAME = f"{INPUT_DATASET_VERSION}__train.csv"
TEST_FILENAME = f"{INPUT_DATASET_VERSION}__test.csv"

# Dataset de salida
OUTPUT_DATASET_VERSION = "CIC17__split__v1__seleccion__v1"
OUTPUT_DIR = f"../../../02_datasets/processed/{OUTPUT_DATASET_VERSION}"

TRAIN_OUTPUT_FILENAME = f"{OUTPUT_DATASET_VERSION}__train.csv"
TEST_OUTPUT_FILENAME = f"{OUTPUT_DATASET_VERSION}__test.csv"

LOADINGS_OUTPUT_FILENAME = f"{OUTPUT_DATASET_VERSION}__pca_loadings.csv"
REPORT_OUTPUT_FILENAME = f"{OUTPUT_DATASET_VERSION}__report.json"

# PCA
N_COMPONENTS_ANALISIS = 5
TOP_N_POR_COMPONENTE = 5

RANDOM_STATE = 42

In [3]:
df_train = cargar_dataset(
    nombre_dataset=TRAIN_FILENAME,
    ruta_base=INPUT_DIR
)

df_test = cargar_dataset(
    nombre_dataset=TEST_FILENAME,
    ruta_base=INPUT_DIR
)

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

display(df_train.head())

Train shape: (2016638, 48)
Test shape: (504160, 48)


,DESTINATION_PORT,FLOW_DURATION,TOTAL_FWD_PACKETS,TOTAL_LENGTH_OF_FWD_PACKETS,FWD_PACKET_LENGTH_MAX,FWD_PACKET_LENGTH_MIN,FWD_PACKET_LENGTH_MEAN,BWD_PACKET_LENGTH_MAX,BWD_PACKET_LENGTH_MIN,FLOW_BYTES_S,...,INIT_WIN_BYTES_FORWARD,INIT_WIN_BYTES_BACKWARD,ACT_DATA_PKT_FWD,MIN_SEG_SIZE_FORWARD,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_STD,LABEL
0,60146,974,6,454,258,0,75.666667,6,6,496919.917900,...,114,0,4,32,0.0,0.0,0,0,0.0,0
1,443,5227824,7,599,517,0,85.571429,152,0,143.654415,...,29200,237,3,32,0.0,0.0,0,0,0.0,0
2,53,49756,1,49,49,49,49.000000,77,77,2532.357907,...,-1,-1,0,32,0.0,0.0,0,0,0.0,0
3,80,170446,3,26,20,0,8.666667,4380,0,68215.153190,...,8192,229,2,20,0.0,0.0,0,0,0.0,2
4,53,170,2,58,29,29,29.000000,45,45,870588.235300,...,-1,-1,1,32,0.0,0.0,0,0,0.0,0


In [4]:
if LABEL_COL not in df_train.columns:
    raise ValueError(f"No existe la columna {LABEL_COL} en train.")

if LABEL_COL not in df_test.columns:
    raise ValueError(f"No existe la columna {LABEL_COL} en test.")

print("Columna LABEL encontrada correctamente.")
print("Columnas train:", df_train.shape[1])
print("Columnas test:", df_test.shape[1])

Columna LABEL encontrada correctamente.
Columnas train: 48
Columnas test: 48


In [5]:
print("Distribución de clases en train:")
display(resumen_clases(df_train, label_col=LABEL_COL))

print("Distribución de clases en test:")
display(resumen_clases(df_test, label_col=LABEL_COL))

Distribución de clases en train:


,count,percentage
LABEL,,
0,1676045,83.1109
1,138277,6.8568
2,102411,5.0783
3,72555,3.5978
4,8229,0.4081
5,4745,0.2353
6,4308,0.2136
7,4182,0.2074
8,2575,0.1277


Distribución de clases en test:


,count,percentage
LABEL,,
0,419012,83.1109
1,34569,6.8568
2,25603,5.0783
3,18139,3.5979
4,2057,0.4080
5,1186,0.2352
6,1077,0.2136
7,1046,0.2075
8,644,0.1277


In [6]:
X_train = df_train.drop(columns=[LABEL_COL])
y_train = df_train[LABEL_COL].copy()

X_test = df_test.drop(columns=[LABEL_COL])
y_test = df_test[LABEL_COL].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (2016638, 47)
y_train: (2016638,)
X_test: (504160, 47)
y_test: (504160,)


In [7]:
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

non_numeric_cols = [
    col for col in X_train.columns
    if col not in numeric_cols
]

print(f"Columnas numéricas usadas en PCA: {len(numeric_cols)}")
print(f"Columnas no numéricas excluidas: {len(non_numeric_cols)}")

if len(non_numeric_cols) > 0:
    print("Columnas no numéricas:")
    print(non_numeric_cols)

Columnas numéricas usadas en PCA: 47
Columnas no numéricas excluidas: 0


In [8]:
missing_in_test = [
    col for col in numeric_cols
    if col not in X_test.columns
]

if len(missing_in_test) > 0:
    raise ValueError(f"Columnas de train que no están en test: {missing_in_test}")

X_train_num = X_train[numeric_cols].copy()
X_test_num = X_test[numeric_cols].copy()

print("X_train_num:", X_train_num.shape)
print("X_test_num:", X_test_num.shape)

X_train_num: (2016638, 47)
X_test_num: (504160, 47)


In [9]:
scaler = RobustScaler()

X_train_scaled = scaler.fit_transform(X_train_num)
X_test_scaled = scaler.transform(X_test_num)

print("RobustScaler aplicado correctamente.")
print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)

RobustScaler aplicado correctamente.
X_train_scaled: (2016638, 47)
X_test_scaled: (504160, 47)


In [10]:
pca = PCA(
    n_components=N_COMPONENTS_ANALISIS,
    random_state=RANDOM_STATE
)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("PCA calculado correctamente.")
print("X_train_pca:", X_train_pca.shape)
print("X_test_pca:", X_test_pca.shape)

PCA calculado correctamente.
X_train_pca: (2016638, 5)
X_test_pca: (504160, 5)


In [11]:
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print("Varianza explicada por componente:")

for i, var in enumerate(explained_variance, start=1):
    print(f"PC{i}: {var:.6f} ({var * 100:.2f}%)")

print()
print(f"Varianza acumulada con las {N_COMPONENTS_ANALISIS} primeras componentes:")
print(f"{cumulative_variance[-1]:.6f} ({cumulative_variance[-1] * 100:.2f}%)")

Varianza explicada por componente:
PC1: 0.913548 (91.35%)
PC2: 0.068625 (6.86%)
PC3: 0.013467 (1.35%)
PC4: 0.002521 (0.25%)
PC5: 0.000931 (0.09%)

Varianza acumulada con las 5 primeras componentes:
0.999092 (99.91%)


In [12]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=numeric_cols,
    columns=[f"PC{i}" for i in range(1, N_COMPONENTS_ANALISIS + 1)]
)

display(loadings.head())

,PC1,PC2,PC3,PC4,PC5
DESTINATION_PORT,6.406713e-08,-2.005389e-06,1.676060e-06,-1.084889e-05,0.000006
FLOW_DURATION,3.270877e-07,1.217615e-06,-1.304292e-06,5.525350e-06,-0.000006
TOTAL_FWD_PACKETS,3.120705e-08,4.101569e-06,4.926138e-06,-1.091340e-06,-0.000004
TOTAL_LENGTH_OF_FWD_PACKETS,1.749533e-07,2.526379e-06,-1.542170e-06,-7.764200e-07,-0.000001
FWD_PACKET_LENGTH_MAX,1.387698e-07,3.583809e-07,7.359827e-10,-4.879962e-07,-0.000001


In [13]:
features_por_componente = {}

for pc in loadings.columns:
    top_features = (
        loadings[pc]
        .abs()
        .sort_values(ascending=False)
        .head(TOP_N_POR_COMPONENTE)
    )
    
    features_por_componente[pc] = top_features.index.tolist()
    
    print(f"\n===== {pc} =====")
    display(top_features.to_frame(name="peso_absoluto"))


===== PC1 =====


,peso_absoluto
IDLE_STD,0.999770
ACTIVE_MAX,0.016462
ACTIVE_MEAN,0.010501
ACTIVE_STD,0.007150
ACTIVE_MIN,0.005212



===== PC2 =====


,peso_absoluto
ACTIVE_MAX,0.789345
ACTIVE_MEAN,0.471502
ACTIVE_MIN,0.329701
ACTIVE_STD,0.213224
IDLE_STD,0.021195



===== PC3 =====


,peso_absoluto
ACTIVE_MIN,0.686620
ACTIVE_STD,0.482934
ACTIVE_MAX,0.385222
ACTIVE_MEAN,0.383286
FWD_IAT_MIN,0.003829



===== PC4 =====


,peso_absoluto
BWD_IAT_MIN,0.710813
FWD_IAT_MIN,0.702363
FLOW_IAT_MIN,0.030341
ACTIVE_STD,0.015183
ACTIVE_MEAN,0.012703



===== PC5 =====


,peso_absoluto
ACTIVE_STD,0.628260
ACTIVE_MEAN,0.602568
ACTIVE_MAX,0.465010
ACTIVE_MIN,0.155100
FWD_IAT_MIN,0.041734


In [14]:
selected_features = []

for pc, features in features_por_componente.items():
    for feature in features:
        if feature not in selected_features:
            selected_features.append(feature)

print(f"Número total de columnas seleccionadas: {len(selected_features)}")
print()

for feature in selected_features:
    print("-", feature)

Número total de columnas seleccionadas: 8

- IDLE_STD
- ACTIVE_MAX
- ACTIVE_MEAN
- ACTIVE_STD
- ACTIVE_MIN
- FWD_IAT_MIN
- BWD_IAT_MIN
- FLOW_IAT_MIN


In [15]:
df_train_reduced = df_train[selected_features + [LABEL_COL]].copy()
df_test_reduced = df_test[selected_features + [LABEL_COL]].copy()

print("Train original:", df_train.shape)
print("Train reducido:", df_train_reduced.shape)

print("Test original:", df_test.shape)
print("Test reducido:", df_test_reduced.shape)

display(df_train_reduced.head())

Train original: (2016638, 48)
Train reducido: (2016638, 9)
Test original: (504160, 48)
Test reducido: (504160, 9)


,IDLE_STD,ACTIVE_MAX,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MIN,FWD_IAT_MIN,BWD_IAT_MIN,FLOW_IAT_MIN,LABEL
0,0.0,0,0.0,0.0,0,1,1,1,0
1,0.0,0,0.0,0.0,0,3,4,3,0
2,0.0,0,0.0,0.0,0,0,0,49756,0
3,0.0,0,0.0,0.0,0,236,180,56,2
4,0.0,0,0.0,0.0,0,48,3,3,0


In [16]:
print("Distribución de clases en train reducido:")
display(resumen_clases(df_train_reduced, label_col=LABEL_COL))

print("Distribución de clases en test reducido:")
display(resumen_clases(df_test_reduced, label_col=LABEL_COL))

Distribución de clases en train reducido:


,count,percentage
LABEL,,
0,1676045,83.1109
1,138277,6.8568
2,102411,5.0783
3,72555,3.5978
4,8229,0.4081
5,4745,0.2353
6,4308,0.2136
7,4182,0.2074
8,2575,0.1277


Distribución de clases en test reducido:


,count,percentage
LABEL,,
0,419012,83.1109
1,34569,6.8568
2,25603,5.0783
3,18139,3.5979
4,2057,0.4080
5,1186,0.2352
6,1077,0.2136
7,1046,0.2075
8,644,0.1277


In [17]:
guardar_dataset_csv(
    df=df_train_reduced,
    nombre_archivo=TRAIN_OUTPUT_FILENAME,
    ruta=OUTPUT_DIR
)

guardar_dataset_csv(
    df=df_test_reduced,
    nombre_archivo=TEST_OUTPUT_FILENAME,
    ruta=OUTPUT_DIR
)

print("Datasets reducidos guardados correctamente:")
print(os.path.join(OUTPUT_DIR, TRAIN_OUTPUT_FILENAME))
print(os.path.join(OUTPUT_DIR, TEST_OUTPUT_FILENAME))

Datasets reducidos guardados correctamente:
../../../02_datasets/processed/CIC17__split__v1__seleccion__v1/CIC17__split__v1__seleccion__v1__train.csv
../../../02_datasets/processed/CIC17__split__v1__seleccion__v1/CIC17__split__v1__seleccion__v1__test.csv


In [18]:
loadings_to_save = loadings.reset_index().rename(columns={"index": "FEATURE"})

guardar_dataset_csv(
    df=loadings_to_save,
    nombre_archivo=LOADINGS_OUTPUT_FILENAME,
    ruta=OUTPUT_DIR
)

print("Loadings guardados correctamente:")
print(os.path.join(OUTPUT_DIR, LOADINGS_OUTPUT_FILENAME))

Loadings guardados correctamente:
../../../02_datasets/processed/CIC17__split__v1__seleccion__v1/CIC17__split__v1__seleccion__v1__pca_loadings.csv


In [19]:
report = {
    "dataset": DATASET_NAME,
    "input_dataset_version": INPUT_DATASET_VERSION,
    "output_dataset_version": OUTPUT_DATASET_VERSION,
    "input_dir": INPUT_DIR,
    "output_dir": OUTPUT_DIR,
    "train_filename": TRAIN_FILENAME,
    "test_filename": TEST_FILENAME,
    "train_output_filename": TRAIN_OUTPUT_FILENAME,
    "test_output_filename": TEST_OUTPUT_FILENAME,
    "label_col": LABEL_COL,
    "n_components_analisis": N_COMPONENTS_ANALISIS,
    "top_n_por_componente": TOP_N_POR_COMPONENTE,
    "random_state": RANDOM_STATE,
    "train_shape_original": list(df_train.shape),
    "test_shape_original": list(df_test.shape),
    "train_shape_reduced": list(df_train_reduced.shape),
    "test_shape_reduced": list(df_test_reduced.shape),
    "num_numeric_cols_used_for_pca": len(numeric_cols),
    "num_non_numeric_cols_excluded": len(non_numeric_cols),
    "non_numeric_cols_excluded": non_numeric_cols,
    "num_selected_features": len(selected_features),
    "selected_features": selected_features,
    "features_por_componente": features_por_componente,
    "explained_variance_ratio": explained_variance.tolist(),
    "cumulative_variance_ratio": cumulative_variance.tolist()
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

report_path = os.path.join(OUTPUT_DIR, REPORT_OUTPUT_FILENAME)

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=4, ensure_ascii=False)

print("Reporte guardado correctamente:")
print(report_path)

Reporte guardado correctamente:
../../../02_datasets/processed/CIC17__split__v1__seleccion__v1/CIC17__split__v1__seleccion__v1__report.json


In [20]:
print("======================================")
print("PREPROCESAMIENTO FINALIZADO")
print("======================================")
print()
print("Dataset de entrada:")
print(INPUT_DATASET_VERSION)
print()
print("Dataset de salida:")
print(OUTPUT_DATASET_VERSION)
print()
print("Train original:", df_train.shape)
print("Train reducido:", df_train_reduced.shape)
print()
print("Test original:", df_test.shape)
print("Test reducido:", df_test_reduced.shape)
print()
print(f"Componentes PCA analizadas: {N_COMPONENTS_ANALISIS}")
print(f"Top variables por componente: {TOP_N_POR_COMPONENTE}")
print(f"Variables finales seleccionadas: {len(selected_features)}")
print()
print("Archivos generados:")
print("-", os.path.join(OUTPUT_DIR, TRAIN_OUTPUT_FILENAME))
print("-", os.path.join(OUTPUT_DIR, TEST_OUTPUT_FILENAME))
print("-", os.path.join(OUTPUT_DIR, LOADINGS_OUTPUT_FILENAME))
print("-", report_path)

PREPROCESAMIENTO FINALIZADO

Dataset de entrada:
CIC17__split__v1

Dataset de salida:
CIC17__split__v1__seleccion__v1

Train original: (2016638, 48)
Train reducido: (2016638, 9)

Test original: (504160, 48)
Test reducido: (504160, 9)

Componentes PCA analizadas: 5
Top variables por componente: 5
Variables finales seleccionadas: 8

Archivos generados:
- ../../../02_datasets/processed/CIC17__split__v1__seleccion__v1/CIC17__split__v1__seleccion__v1__train.csv
- ../../../02_datasets/processed/CIC17__split__v1__seleccion__v1/CIC17__split__v1__seleccion__v1__test.csv
- ../../../02_datasets/processed/CIC17__split__v1__seleccion__v1/CIC17__split__v1__seleccion__v1__pca_loadings.csv
- ../../../02_datasets/processed/CIC17__split__v1__seleccion__v1/CIC17__split__v1__seleccion__v1__report.json
